In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from gdynia_srodmiescie.loading import load_df_short, load_df_long
from gdynia_srodmiescie import building_features as bf
from gdynia_srodmiescie import street_metrics as sm
from gdynia_srodmiescie import styling as st

In [ ]:
short = load_df_short("../../data/processed/fixed/srodmiescie_short.csv")
long = load_df_long("../../data/processed/fixed/srodmiescie_long_categories.csv")
buildings = (
    short.pipe(bf.add_building_floor_flags, long)
    .pipe(bf.add_building_type)
    .pipe(bf.add_building_periods)
    .pipe(bf.add_vacancy_kind, long)
)

crosstab street x category

In [ ]:
pd.crosstab(
    long.reset_index()["addr_street"],
    long.reset_index()["category"],
    normalize="columns",
).pipe(st.background_gradient, axis="index").pipe(st.percent_formatter)

crosstab time period x type of building

In [ ]:
pd.crosstab(
    buildings.reset_index()["time_period"],
    buildings.reset_index()["type"],
    normalize="columns",
).pipe(st.background_gradient, axis="index").pipe(st.percent_formatter)

In [ ]:
columns_for_corr_1 = [
    "num_premises_wo_vacant",
    "func_entropy",
    "share_vacant",
    "share_mix_use",
    "leader_share",
]
street_metr = long.pipe(sm.street_metrics, buildings)
street_metr.loc[:, columns_for_corr_1].corr(method="spearman")

In [ ]:
columns_for_corr_2 = [
    "num_premises_wo_vacant",
    "func_entropy",
]
street_metr.loc[lambda d: d.index != "Świętojańska", columns_for_corr_2].corr(
    method="pearson"
)

In [ ]:
street_metr.loc[:, columns_for_corr_2].corr(method="pearson")

before vs after

In [ ]:
old_ids = short.loc[
    lambda e: e["date_of_checking"] < pd.to_datetime("2026-10-22")
].index

old_long = long.loc[lambda d: d.index.isin(old_ids)]
old_buildings = buildings.loc[lambda d: d.index.isin(old_ids)]

street_metr_old = old_long.pipe(sm.street_metrics, old_buildings)

func_entropy_before_after = pd.concat(
    {
        "before": street_metr_old.loc[:, columns_for_corr_2].assign(
            rank_before=lambda d: d["func_entropy"].rank(
                method="dense", ascending=False
            )
        ),
        "after": street_metr.loc[:, columns_for_corr_2].assign(
            rank_after=lambda d: d["func_entropy"].rank(method="dense", ascending=False)
        ),
    },
    axis=1,
)


func_entropy_before_after.dropna().sort_values(("after", "rank_after")).loc[
    :, ("before", ["num_premises_wo_vacant", "func_entropy"])
].corr(method="spearman")

In [ ]:
func_entropy_before_after.dropna().sort_values(("after", "rank_after")).loc[
    :, ("after", ["num_premises_wo_vacant", "func_entropy"])
].corr(method="spearman")

In [ ]:
func_entropy_before_after.dropna().sort_values(("after", "rank_after")).loc[
    :, [("after", "func_entropy"), ("before", "func_entropy")]
].corr(method="spearman")

Num of all share how much of all premises are in mixed-use buildings

In [ ]:
func_ground_floor = ["mixed-use", "service"]

ground_profiles = long.pipe(sm.ground_floor_profile, buildings)
ground_profiles.loc[:, func_ground_floor].pipe(
    st.background_gradient, axis="columns"
).pipe(st.percent_formatter)

vacancy vs building condition

In [ ]:
pd.crosstab(buildings["condition"], buildings["is_vacant"], normalize="index").pipe(
    st.background_gradient
).pipe(st.percent_formatter)

vacancy kind vs building condition

In [ ]:
pd.crosstab(buildings["condition"], buildings["vacancy_kind"], normalize="index").pipe(
    st.background_gradient
).pipe(st.percent_formatter)

vacancy kind vs time period

In [ ]:
pd.crosstab(
    buildings["vacancy_kind"], buildings["time_period"], normalize="columns"
).pipe(st.background_gradient).pipe(st.percent_formatter)

streets with vacancy among working

In [ ]:
pd.crosstab(
    buildings["addr_street"],
    buildings["vacancy_kind"].eq("vacant among working"),
    normalize="index",
).rename(columns={False: "no vacant among working", True: "vacant among working"}).join(
    pd.crosstab(
        buildings["addr_street"],
        buildings["vacancy_kind"].eq("vacant among working"),
        margins=True,
    ).rename(columns={"All": "sum_build_per_street"})["sum_build_per_street"]
).loc[lambda d: d["vacant among working"].gt(0)].pipe(
    st.background_gradient, subset="vacant among working"
).pipe(st.percent_formatter, subset=["no vacant among working", "vacant among working"])

corr type entropy x func entropy

In [ ]:
types_for_corr_3 = ["type_entropy", "func_entropy", "num_premises_wo_vacant"]

building_type_shares = buildings.pipe(sm.building_type_shares)
type_entropy = building_type_shares.pipe(sm.entropy_from_shares)
street_metr_type_entropy = street_metr.join(type_entropy)
street_metr_type_entropy.loc[:, types_for_corr_3].corr(method="spearman")

In [ ]:
building_type_shares.join(type_entropy["type_entropy"]).loc[
    street_metr.index
].sort_values("type_entropy", ascending=False).pipe(
    st.background_gradient, subset=building_type_shares.columns
).pipe(st.percent_formatter, subset=building_type_shares.columns).pipe(
    st.decimal_formatter, decimals=3, subset="type_entropy"
)

In [ ]:
street_metr_type_entropy.sort_values("func_entropy", ascending=False).iloc[2:].loc[
    :, ["func_entropy", "type_entropy"]
].corr(method="spearman")

In [ ]:
long.pipe(sm.street_metrics, buildings, cat_omitted=bf.NON_SERVICE).join(
    type_entropy
).loc[:, ["func_entropy", "type_entropy"]].corr(method="spearman")

In [ ]:
short.loc[lambda d: d.index.isin(old_ids)].to_csv(
    "../../tests/data/short_old.csv", sep=";"
)
long.loc[lambda d: d.index.isin(old_ids)].to_csv(
    "../../tests/data/long_old.csv", sep=";"
)

In [ ]:
street_metr

In [ ]:

streets_to_show = ['Armii Krajowej', '3 Maja', 'Portowa', 'Antoniego Abrahama', 'Starowiejska',
                   'Świętojańska', 'Obrońców Wybrzeża', 'Antoniego Hryniewickiego', 'Władysława IV']

street_metr = street_metr.assign(
    label = lambda d: d.index.to_series().where(d.index.isin(streets_to_show))
)

st.plot_scatter(street_metr,'num_premises_wo_vacant', 'func_entropy', "scatter1.png", label_name= 'label', xscale= "log")

In [ ]:
street_metr